## This notebook is part of the Apache Spark training delivered by CERN IT


Run this notebook from Databricks with Python kernel

### SPARK DataFrame Hands-On Lab
Modified for Databricks, original content from CERN's GitHub repo:
https://github.com/cerndb/SparkTraining
Contact: Luca.Canali@cern.ch

Forked to a personal GitHub account at:
https://github.com/Muhammad-Umer-Ali/SparkTraining

### Objective: Perform Basic DataFrame Operations
1. Creating DataFrames
2. Select columns
3. Add, rename and drop columns
4. Filtering rows
5. Aggregations

## Getting started: create the SparkSession
Following code is maintained for legacy spark usage, or on-prem clusters. This is not needed in Databricks Notebooks because `SparkSession` is already made available in the variable named `spark` by the Databricks environment.

Notice in the following cell that the code cells can be disabled in the notebook, if required.

In [0]:
%skip
# !pip install pyspark

#from pyspark.sql import SparkSession

#spark = (SparkSession.builder
#          .master("local[*]") \
#          .appName("DataFrame HandsOn 1") \
#          .config("spark.ui.showConsoleProgress","false") \
#          .getOrCreate()
#        )

spark

The master `local[*]` means that the executors are in the same node that is running the driver. The `*` tells Spark to start as many executors as there are logical cores available

### Hands-On 1 - Construct a DataFrame from csv file
This demostrates how to read a csv file and construct a DataFrame.  
We will use the online retail dataset from Kaggle, credits: https://www.kaggle.com/datasets/vijayuv/onlineretail


#### First, let's inspect the csv content

In [0]:
!gzip -cd ../data/online-retail-dataset.csv.gz 2>&1| head -n3

In [0]:
online_retail_schema="InvoiceNo int, StockCode string, Description string, Quantity int,\
InvoiceDate timestamp,UnitPrice float,CustomerId int, Country string"

In [0]:
df = (spark.read
        .option("header", "true")
        .option("timestampFormat", "M/d/yyyy H:m")
        .csv("../data/online-retail-dataset.csv.gz",
             schema=online_retail_schema)
     )

## Lazy Evaluation
The next code cell will fail on execution. Notice the **Lazy Evaluation** in play. The failure didn't happen when we specified the path of the input data. Actual access to data and the processing happened when we demanded some output by calling a Spark **Action**.

#### Resolve the Error and Inspect the data

In [0]:
df.show(2, False)

Correct Version of the cell above (for reference):

In [0]:
df = (spark.read
        .option("header", "true")
        .option("timestampFormat", "M/d/yyyy H:m")
        .csv("/Workspace/Shared/Cern Spark Training/data/online-retail-dataset.csv.gz",
             schema=online_retail_schema)
     )

#### Show columns

In [0]:
df.printSchema()

### Hands-On 2 - Spark Transformations - select, add, rename and drop columns

Select dataframe columns

In [0]:
# select single column

df.select("Country").show(2)

Select multiple columns


In [0]:
df.select("StockCode","Description","UnitPrice").show(n=2, truncate=False)

In [0]:
df.columns

### Display() vs Show()

In [0]:
# select first 5 columns
df.select(df.columns[0:5]).display()

In [0]:
# selects all the original columns and adds a new column that specifies high value item
df.selectExpr(
   "*", # all original columns
   "(UnitPrice > 100) as HighValueItem")

In [0]:
df.display()

In [0]:
# selects all the original columns and adds a new column that specifies high value item
(df.selectExpr(
  "sum(Quantity) as TotalQuantity",
  "cast(sum(UnitPrice) as int) as InventoryValue")
  .display()
)

#### Adding, renaming and dropping columns

In [0]:
# add a new column called InvoiceValue
from pyspark.sql.functions import expr
df_1 = (df
        .withColumn("InvoiceValue", expr("UnitPrice * Quantity"))
        .select("InvoiceNo","Description","UnitPrice","Quantity","InvoiceValue")
       )
df_1.limit(2).display()

# rename InvoiceValue to LineTotal
df_2 = df_1.withColumnRenamed("InvoiceValue","LineTotal")
df_2.limit(2).display()

# drop a column
df_2.drop("LineTotal").limit(2).display()

### Hands-On 3 - Spark Transformations - filter, sort and cast

In [0]:
from pyspark.sql.functions import col

# select invoice lines with quantity > 50 and unitprice > 20
df.where(col("Quantity") > 20).where(col("UnitPrice") > 50).display()
df.filter(df.Quantity > 20).filter(df.UnitPrice > 50).display()
df.filter("Quantity > 20 and UnitPrice > 50").display()

In [0]:
# select invoice lines with quantity > 100 or unitprice > 20
df.where((col("Quantity") > 100) | (col("UnitPrice") > 20)).limit(2).display()

In [0]:
from pyspark.sql.functions import desc, asc

# sort in the default order: ascending
df.orderBy(expr("UnitPrice")).limit(5).display()

df.orderBy(col("Quantity").desc(), col("UnitPrice").asc()).limit(20).display()

### Hands-On 4 - Spark Transformations - aggregations
full list of built int functions - https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql.html#functions

In [0]:
%%time
# Count distinct customers
from pyspark.sql.functions import countDistinct
df.select(countDistinct("CustomerID")).display()

In [0]:
%%time
# approx. distinct stock items
from pyspark.sql.functions import approx_count_distinct
df.select(approx_count_distinct("CustomerID", 0.1)).display()

In [0]:
# average, maximum and minimum purchase quantity
from pyspark.sql.functions import avg, max, min
( df.select(
    avg("Quantity").alias("avg_purchases"),
    max("Quantity").alias("max_purchases"),
    min("Quantity").alias("min_purchases"))
   .display()
)

### Hands-On 5 - Spark Transformations - grouping and windows

In [0]:
# count of items on the invoice
df.groupBy("InvoiceNo", "CustomerId").count().limit(5).display()

# grouping with expressions
df.groupBy("InvoiceNo").agg(expr("avg(Quantity)"),expr("stddev_pop(Quantity)"))\
  .limit(5).display()

### Read the csv file into DataFrame

`%%time` is an iPython magic https://ipython.readthedocs.io/en/stable/interactive/magics.html


It's possible to read files without specifying the schema. Some file formats (Parquet is one of them) include the schema, which means that Spark can start reading the file. For format without schema (csv, json...) Spark can infer the schema. Let's see what's the difference in terms of time and of results:

In [0]:
online_retail_schema="InvoiceNo int, StockCode string, Description string, Quantity int,\
InvoiceDate timestamp,UnitPrice float,CustomerId int, Country string"

In [0]:
%%time
df = spark.read \
        .option("header", "true") \
        .option("timestampFormat", "M/d/yyyy H:m")\
        .csv("../data/online-retail-dataset.csv.gz",
             schema=online_retail_schema)

In [0]:
%%time
df_infer = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv("/Workspace/Shared/Cern Spark Training/data/online-retail-dataset.csv.gz")

## Transition From PySpark to SparkSQL

From PySpark code, `spark.sql()` method can be invoked to create new Dataframes by calling SQL directly. But you need to register the dataframe as a *Temporary View* first so it can be accessed in the SQL query. Temporary Views are present only in the context of the Spark session so they are not preserved in Unity Catalog. For the same reason, they cannot be accessed by anyone outside of current application; in our case this

In [0]:
df.createOrReplaceTempView("TempDF")

summary=spark.sql("SELECT StockCode, SUM(try_cast(Quantity as int)) AS Total_Quantity FROM TempDF GROUP BY StockCode")

display(summary)

Databricks visualization. Run in Databricks to view.

## Exercises

Reminder: documentation at 
https://spark.apache.org/docs/latest/api/python/index.html

In [0]:
online_retail_schema="InvoiceNo int, StockCode string, Description string, Quantity int,\
InvoiceDate timestamp,UnitPrice float,CustomerId int, Country string"

df = spark.read \
        .option("header", "true") \
        .option("timestampFormat", "M/d/yyyy H:m")\
        .csv("../data/online-retail-dataset.csv.gz",
             schema=online_retail_schema)

Task: Show 5 lines of the "description" column 

Task: Count the number of distinct invoices in the dataframe

Task: Find out in which month most invoices have been issued

Task: Filter the lines where the Quantity is more than 30

Task: Show the four most sold items (by quantity)

Bonus question: why do these two operations return different results? Hint: look at the documentation 

In [0]:
print(df.select("InvoiceNo").distinct().count())
from pyspark.sql.functions import countDistinct
df.select(countDistinct("InvoiceNo")).display()